# Análise Financeira com Python

## Desafio Final - Fintech ClearBank

Este notebook foi desenvolvido como solução para o desafio de análise de dados financeiros da fintech **ClearBank**.
O objetivo é ler um arquivo de transações bancárias (`transacoes.csv`), realizar a validação e limpeza dos dados, calcular métricas financeiras agrupadas por mês, identificar transações financeiras potencialmente suspeitas, exibir um relatório formatado e salvar os resultados estruturados em formato JSON.

### Estrutura das Colunas do Arquivo de Entrada (`transacoes.csv`):

| Coluna | Tipo esperado | Descrição |
| :--- | :--- | :--- |
| **id** | inteiro | Identificador único da transação |
| **data** | texto | Data no formato `AAAA-MM-DD` |
| **cliente_id** | texto | Código do cliente (não pode ser vazio) |
| **tipo** | texto | `credito` ou `debito` |
| **valor** | decimal | Valor da transação (deve ser maior que 0) |
| **descricao** | texto | Descrição livre da operação |
| **categoria** | texto | Ex.: `salario`, `compra`, `transferencia` |

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### ⚠️ Célula de Instruções Importantes

Antes de executar as células de código deste notebook, certifique-se de que:
1. O arquivo `transacoes.csv` está criado no mesmo diretório deste notebook.
2. O arquivo `transacoes.csv` contém:
   - Pelo menos **15 registros válidos** distribuídos em **3 ou mais meses** distintos.
   - Pelo menos **5 registros inválidos** para testar as regras de validação (por exemplo: valores não numéricos, ids vazios, datas incorretas, tipos de transação inválidos, etc.).
   - Pelo menos **2 transações** com valor acima de **R$ 10.000,00** para testar a sinalização de transações suspeitas.

O script foi construído para lidar silenciosamente com linhas corrompidas ou inválidas, descartando-as sem interromper o fluxo do processamento e contabilizando-as no resumo de limpeza final.

### 1. Importações de Bibliotecas e Constantes

Utilizaremos exclusivamente módulos nativos do Python:
- `csv` para a manipulação e leitura do arquivo de transações.
- `json` para exportar o relatório financeiro consolidado.
- `datetime` (do módulo `datetime`) para manipulação de datas e cálculo de períodos.

In [2]:
import csv
import json
from datetime import datetime

LIMITE_SUSPEITO = 10000.00
arquivo_csv = '/content/drive/MyDrive/transacoes.csv'

### 2. Função Auxiliar de Formatação Monetária

A função `formatar_moeda` converte valores numéricos em strings elegantes no padrão de moeda brasileiro (`R$ X.XXX,XX`), facilitando a leitura dos relatórios.

In [3]:
def formatar_moeda(valor):
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

### 3. Validar transação

Atua como um filtro de qualidade para os dados brutos. Ela recebe uma única linha do CSV e verifica se todas as regras de negócio foram atendidas (ID numérico, cliente preenchido, tipo válido, valor positivo e data no padrão correto). Se a linha for perfeita, ela converte os dados para os tipos adequados (inteiro, float, datetime) e a retorna limpa. Se houver qualquer erro, ela retorna None, sinalizando que a linha deve ser descartada silenciosamente

In [11]:
def validar_transacao(linha):
    if not linha.get('id') or not linha['id'].isdigit():
        return None

    if not linha.get('cliente_id') or linha['cliente_id'].strip() == "":
        return None

    if linha.get('tipo') not in ['credito', 'debito']:
        return None

    try:
        valor = float(linha.get('valor', 0))
        if valor <= 0:
            return None
        linha['valor'] = valor
    except ValueError:
        return None

    try:
        linha['data'] = datetime.strptime(linha.get('data', ''), "%Y-%m-%d")
    except ValueError:
        return None

    linha['id'] = int(linha['id'])
    return linha

## 4. Ler transações

É a função responsável pelo carregamento inicial dos dados. Ela abre o arquivo CSV utilizando o módulo nativo csv.DictReader. Em seguida, percorre linha por linha, repassando cada uma para a validar_transacao(). No final, ela entrega uma lista contendo apenas as transações válidas, além de contabilizar quantas linhas foram lidas no total e quantas falharam.

In [12]:
def ler_transacoes(nome_arquivo):
    transacoes_validas = []
    qtd_invalidas = 0
    total_lidas = 0

    try:
        with open(nome_arquivo, mode='r', encoding='utf-8') as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                total_lidas += 1
                transacao_limpa = validar_transacao(linha)

                if transacao_limpa:
                    transacoes_validas.append(transacao_limpa)
                else:
                    qtd_invalidas += 1
    except FileNotFoundError:
        print(f"Erro: O arquivo '{nome_arquivo}' não foi encontrado.")

    return transacoes_validas, qtd_invalidas, total_lidas

## 5. Gerar relatório

Esta função recebe a lista de transações limpas e faz o agrupamento mês a mês. Ela calcula todas as métricas exigidas: quantidade de transações, soma de créditos e débitos, saldo, valor médio, além de identificar a maior e a menor transação de cada mês. Simultaneamente, ela fiscaliza os valores e separa qualquer movimentação que ultrapasse o limite suspeito estipulado

In [13]:
def gerar_relatorio(transacoes):
    transacoes_suspeitas = []
    resumo_mensal = {}

    if not transacoes:
        return resumo_mensal, transacoes_suspeitas, 0, None, None

    data_mais_antiga = transacoes[0]['data']
    data_mais_recente = transacoes[0]['data']

    for t in transacoes:
        if t['data'] < data_mais_antiga:
            data_mais_antiga = t['data']
        if t['data'] > data_mais_recente:
            data_mais_recente = t['data']

        if t['valor'] > LIMITE_SUSPEITO:
            transacoes_suspeitas.append({
                "id": t['id'],
                "cliente_id": t['cliente_id'],
                "data": t['data'].strftime("%Y-%m-%d"),
                "valor": t['valor']
            })

        mes_ano = t['data'].strftime("%Y-%m")

        if mes_ano not in resumo_mensal:
            resumo_mensal[mes_ano] = {
                "quantidade": 0, "total_credito": 0.0, "total_debito": 0.0,
                "saldo": 0.0, "media": 0.0,
                "maior_valor": t['valor'], "menor_valor": t['valor']
            }

        mes = resumo_mensal[mes_ano]
        mes["quantidade"] += 1

        if t['tipo'] == 'credito':
            mes["total_credito"] += t['valor']
        else:
            mes["total_debito"] += t['valor']

        if t['valor'] > mes["maior_valor"]:
            mes["maior_valor"] = t['valor']
        if t['valor'] < mes["menor_valor"]:
            mes["menor_valor"] = t['valor']

    for mes, dados in resumo_mensal.items():
        dados["saldo"] = dados["total_credito"] - dados["total_debito"]
        if dados["quantidade"] > 0:
            dados["media"] = (dados["total_credito"] + dados["total_debito"]) / dados["quantidade"]

    dias_intervalo = (data_mais_recente - data_mais_antiga).days

    return resumo_mensal, transacoes_suspeitas, dias_intervalo, data_mais_antiga, data_mais_recente

## 6. Apresentar relatório

Exibe os resultados formatados no terminal.

In [14]:
def exibir_relatorio(total_lidas, qtd_validas, qtd_invalidas, resumo_mensal, transacoes_suspeitas, dias, data_inicio, data_fim):
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {qtd_validas}")
    print(f"Linhas inválidas: {qtd_invalidas}")

    if qtd_validas > 0:
        print(f"Período analisado: {data_inicio.strftime('%d/%m/%Y')} → {data_fim.strftime('%d/%m/%Y')} ({dias} dias)")

    print("\n===== RELATÓRIO MENSAL =====")
    for mes, dados in sorted(resumo_mensal.items()):
        print(f"\nMês: {mes}")
        print(f"  Transações:    {dados['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
        print(f"  Média:         {formatar_moeda(dados['media'])}")
        print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")

    print("\n===== TRANSAÇÕES SUSPEITAS =====")
    if transacoes_suspeitas:
        for t in transacoes_suspeitas:
            print(f"ID: {t['id']} | Cliente: {t['cliente_id']} | Data: {t['data']} | Valor: {formatar_moeda(t['valor'])}")
    else:
        print("Nenhuma transação suspeita encontrada.")

## 7. Exportar JSON

Exporta os dados para um arquivo JSON.

In [15]:
def salvar_json(resumo_mensal, transacoes_suspeitas, qtd_validas, qtd_invalidas, nome_arquivo="relatorio.json"):
    dados_exportacao = {
        "gerado_em": datetime.now().strftime("%Y-%m-%d"),
        "total_transacoes_validas": qtd_validas,
        "total_transacoes_invalidas": qtd_invalidas,
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": transacoes_suspeitas
    }
    try:
        with open(nome_arquivo, "w", encoding="utf-8") as arquivo:
            json.dump(dados_exportacao, arquivo, ensure_ascii=False, indent=2)
        print(f"\nRelatório salvo com sucesso em '{nome_arquivo}'!")
    except Exception as e:
        print(f"\nErro ao salvar JSON: {e}")

## Execução

In [19]:
transacoes_validas, qtd_invalidas, total_lidas = ler_transacoes(arquivo_csv)
qtd_validas = len(transacoes_validas)

if qtd_validas > 0:
    resumo_mensal, transacoes_suspeitas, dias, data_inicio, data_fim = gerar_relatorio(transacoes_validas)
    exibir_relatorio(total_lidas, qtd_validas, qtd_invalidas, resumo_mensal, transacoes_suspeitas, dias, data_inicio, data_fim)
    salvar_json(resumo_mensal, transacoes_suspeitas, qtd_validas, qtd_invalidas)
else:
    print("Não há dados válidos para gerar o relatório.")

Total de linhas lidas: 22
Linhas válidas: 16
Linhas inválidas: 6
Período analisado: 05/01/2026 → 15/04/2026 (100 dias)

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações:    5
  Total crédito: R$ 15.500,00
  Total débito:  R$ 550,50
  Saldo:         R$ 14.949,50
  Média:         R$ 3.210,10
  Maior valor:   R$ 12.000,00
  Menor valor:   R$ 50,00

Mês: 2026-02
  Transações:    5
  Total crédito: R$ 18.500,00
  Total débito:  R$ 669,90
  Saldo:         R$ 17.830,10
  Média:         R$ 3.833,98
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações:    5
  Total crédito: R$ 5.500,00
  Total débito:  R$ 1.825,00
  Saldo:         R$ 3.675,00
  Média:         R$ 1.465,00
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 75,00

Mês: 2026-04
  Transações:    1
  Total crédito: R$ 0,00
  Total débito:  R$ 50,00
  Saldo:         R$ -50,00
  Média:         R$ 50,00
  Maior valor:   R$ 50,00
  Menor valor:   R$ 50,00

===== TRANSAÇÕES SUSPEITAS =====
ID: 5 | Client